In [2]:
import pandas as pd

# Đường dẫn dataset
DATA_PATH = r"D:\Subject\Finance Risk\multimodal-financial-risk-intelligence\backend\Data\unemployment_rate\unemployment_rate.csv"

# Load data
df = pd.read_csv(DATA_PATH)

# Chuẩn hóa
df["date"] = pd.to_datetime(df["date"])
df["unemployment_rate"] = pd.to_numeric(
    df["unemployment_rate"],
    errors="coerce"
)

df = (
    df.dropna(subset=["date", "unemployment_rate"])
      .sort_values("date")
      .reset_index(drop=True)
)

# Chừa 24 tháng cuối làm test
TEST_SIZE = 24

train = df.iloc[:-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()

print("Full data:", df.shape)
print("Train:", train.shape)
print("Test:", test.shape)

print("\nTest period:")
print(test[["date", "unemployment_rate"]].to_string(index=False))

Full data: (943, 2)
Train: (919, 2)
Test: (24, 2)

Test period:
      date  unemployment_rate
2024-08-01                4.2
2024-09-01                4.1
2024-10-01                4.1
2024-11-01                4.2
2024-12-01                4.1
2025-01-01                4.0
2025-02-01                4.2
2025-03-01                4.2
2025-04-01                4.2
2025-05-01                4.3
2025-06-01                4.1
2025-07-01                4.3
2025-08-01                4.3
2025-09-01                4.4
2025-11-01                4.5
2025-12-01                4.4
2026-01-01                4.3
2026-02-01                4.4
2026-03-01                4.3
2026-04-01                4.3
2026-05-01                4.3
2026-06-01                4.2
2026-07-01                4.1
2026-08-01                4.1


In [3]:
import torch
from chronos import Chronos2Pipeline

MODEL_NAME = "amazon/chronos-2"

pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
)

# Train data → tensor shape: (batch, variates, history)
context = torch.tensor(
    train["unemployment_rate"].values,
    dtype=torch.float32
).reshape(1, 1, -1)

# Forecast 24 tháng
forecast_output = pipeline.predict(
    context,
    prediction_length=24
)

print("Forecast output type:", type(forecast_output))
print("Forecast tensor shape:", forecast_output[0].shape)

ModuleNotFoundError: No module named 'chronos'

In [ ]:
forecast_tensor = forecast_output[0]

# Shape: (1, 21, 24)
# Lấy series đầu tiên, sau đó median trên 21 forecast samples
prediction = forecast_tensor[0].median(dim=0).values

prediction = prediction.cpu().numpy()

print("Prediction shape:", prediction.shape)
print("Prediction:", prediction)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 6))

plt.plot(
    test["date"],
    test["unemployment_rate"],
    marker="o",
    linewidth=2,
    label="Actual"
)

plt.plot(
    test["date"],
    prediction,
    marker="o",
    linewidth=2,
    label="Chronos-2 Prediction"
)

plt.title(
    "US Unemployment Rate: Actual vs Chronos-2 Prediction",
    fontsize=16
)

plt.xlabel("Date")
plt.ylabel("Unemployment Rate (%)")

plt.xticks(
    test["date"],
    test["date"].dt.strftime("%Y-%m"),
    rotation=45
)

plt.legend()
plt.tight_layout()
plt.show()